In [1]:
import sys
import pandas as pd
from pathlib import Path

# This moves from notebooks/ up one level to project root
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

print("Using project root:", PROJECT_ROOT)


Using project root: C:\Users\nathan.taylor\Jupyter Notebooks\outlier_pipeline


In [2]:
from src.runner import run_all_jobs

CONFIG_PATH = PROJECT_ROOT / "configs" / "job_1_test.yml"

results = run_all_jobs(str(CONFIG_PATH))


2026-02-12 10:48:03,042 [INFO] src.runner - Starting job: VZW_SOL_trust_building
2026-02-12 10:48:03,053 [INFO] pyhive.presto - SELECT 
--CAST(week_stop_date AS DATE) AS week_,
--date,
expert_id,
-- year_month,
metric,
icp_client,
--tenure_group,
site,
SUM(numerator) AS num,
SUM(denominator) AS den,
ROUND(
COALESCE(
CAST(SUM(numerator) AS DOUBLE) /
NULLIF(CAST(SUM(denominator) AS DOUBLE), 0.0),
0.000
),
3
) AS calc
FROM 
hive.care.expert_performance_metrics a
LEFT OUTER JOIN 
hive.care.l4_asurion_umt_ppx_pay_calendar d ON a."date" = CAST(d.event_date AS DATE)
WHERE 
LOWER(metric) = 'erp'
AND icp_client = 'PSS-Verizon'
AND a."date" between DATE '2026-01-30' and DATE '2026-02-13' 
 --and expert_id in ()
GROUP BY 1, 2, 3, 4


2026-02-12 10:48:15,291 [INFO] src.runner - [VZW_SOL_trust_building] Raw query rows: 2276
2026-02-12 10:48:15,299 [INFO] src.runner - [VZW_SOL_trust_building] After validity filters rows: 2070
2026-02-12 10:48:15,315 [INFO] src.runner - [VZW_SOL_trust_building] After

In [3]:
print("\n===== JOB SUMMARY =====\n")

summary_df = pd.DataFrame([
    {
        "job_name": job_name,
        "rows_raw": meta.get("rows_raw"),
        "rows_valid": meta.get("rows_valid"),
        "rows_eligible": meta.get("rows_eligible"),
        "rows_out": meta.get("rows_out"),
        "status": meta.get("status"),
        "output_file": meta.get("out_path"),
    }
    for job_name, meta in results.items()
])

summary_df



===== JOB SUMMARY =====



,job_name,rows_raw,rows_valid,rows_eligible,rows_out,status,output_file
0,VZW_SOL_trust_building,2276,2070,1862,376,success,C:\Users\nathan.taylor\Asurion\Coaching & Tech...


In [4]:
job_key = "VZW_SOL_trust_building"   # or whatever key appears in results
df_job = results[job_key]["df"]

# show whether the flag column exists
print("flag col present:", "cancellation_rate__is_outlier" in df_job.columns)

# counts of the flag (should be only True rows if you filtered)
if "cancellation_rate__is_outlier" in df_job.columns:
    print(df_job["cancellation_rate__is_outlier"].value_counts(dropna=False))


flag col present: True
cancellation_rate__is_outlier
False    308
True      68
Name: count, dtype: int64


In [10]:
import inspect
import src.runner

print("Has output shaping logic?:",
      "filter_to_comparison_outliers" in inspect.getsource(src.runner.run_job))

Has output shaping logic?: True


In [12]:
meta = results["VZW_SOL_trust_building"]
print("rows_out:", meta.get("rows_out"))

rows_out: 376


In [14]:
import yaml
from pathlib import Path

cfg = yaml.safe_load(Path(CONFIG_PATH).read_text(encoding="utf-8-sig"))
job = next(j for j in cfg["jobs"] if j["name"] == "VZW_SOL_trust_building")

print("Top-level output:", job.get("output"))
print("Top-level keys:", sorted(job.keys()))

Top-level output: None
Top-level keys: ['coaching_override', 'comparisons', 'eligibility', 'inputs', 'name', 'query', 'reference_link', 'selection', 'validity']


In [16]:
print(results.keys())

dict_keys(['VZW_SOL_trust_building'])
